# Functions and classes

## Learning objectives

By the end of this notebook you will be able to:

- define functions with parameters, defaults, and docstrings;
- accept a variable number of arguments with `*args` and `**kwargs`;
- write short anonymous functions with `lambda` and use them as sort keys;
- define a class with an initialiser, methods, a property, and `__repr__`;
- explain when a function is enough and when a class earns its keep.

## Concept

A script that repeats itself is hard to trust. **Functions** name a reusable piece of logic and
**classes** bundle related data with the behaviour that belongs to it. This notebook builds both
on top of the penguins table, moving from a single helper to a small object model.

A **function** takes inputs, does one job, and returns a result. Good functions have a
descriptive name, a docstring that states the contract, and no hidden dependence on global
state. Default arguments make common cases short without forcing callers to pass every value.

A `lambda` is a function expression for tiny one-off jobs. It is most at home where another
function expects a callable, such as the `key=` argument of `sorted`.

A **class** is a blueprint for objects. `__init__` sets up each instance, ordinary methods act
on it, `@property` exposes a computed value like an attribute, and `__repr__` gives a readable
description used in the REPL and in f-strings. Classes are worth the extra machinery when the
same few values travel together through several functions — the `Penguin` object below is the
textbook case of "data plus the rules about that data".

## Worked example

### Functions with defaults and a docstring

In [ ]:
from ds_practice import load_penguins

penguins = load_penguins()
rows = penguins.dropna(subset=["body_mass_g"]).to_dict("records")
print("clean rows:", len(rows))
rows[0]

In [ ]:
def mass_band(grams, light=3500, heavy=4500):
    """Return 'light', 'medium', or 'heavy' for a mass in grams."""
    if grams < light:
        return "light"
    if grams < heavy:
        return "medium"
    return "heavy"

print(mass_band(3200), mass_band(4000), mass_band(5000))
print("custom thresholds:", mass_band(4000, light=3000, heavy=3900))

### Variable arguments

`*args` collects positional extras into a tuple and `**kwargs` collects keyword extras into a
dict. This is how a function can stay flexible without a long parameter list.

In [ ]:
def describe(label, *values, unit="g", **extra):
    """Summarise any number of measurements under one label."""
    if not values:
        raise ValueError("describe needs at least one value")
    summary = {
        "n": len(values),
        "mean": round(sum(values) / len(values), 1),
        "unit": unit,
    }
    summary.update(extra)
    return label, summary

print(describe("sample", 3750, 3800, 5000, unit="g", island="Biscoe"))

### Lambdas as sort keys

We often want the five heaviest penguins without writing a named function. A `lambda` expresses
the key inline.

In [ ]:
top_five = sorted(rows, key=lambda row: row["body_mass_g"], reverse=True)[:5]
for row in top_five:
    print(f"{row['species']:<10} {row['body_mass_g']:>5} g  flipper {row['flipper_length_mm']} mm")

# the same idea with a named key function, which is easier to reuse and test
def flipper_length(row):
    return row["flipper_length_mm"]

shortest = min(rows, key=flipper_length)
print("\nshortest flipper:", shortest["species"], shortest["flipper_length_mm"], "mm")

### A class: data plus behaviour

The class stores the fields that belong together and exposes the derived `band` as a property.
`from_row` is a small factory that builds an instance from one dictionary, which is exactly how
a CSV row arrives.

In [ ]:
class Penguin:
    """One penguin measurement with a derived mass band."""

    def __init__(self, species, island, body_mass_g, flipper_length_mm):
        self.species = species
        self.island = island
        self.body_mass_g = float(body_mass_g)
        self.flipper_length_mm = float(flipper_length_mm)

    @classmethod
    def from_row(cls, row):
        """Build a Penguin from a mapping such as one CSV row."""
        return cls(
            species=row["species"],
            island=row["island"],
            body_mass_g=row["body_mass_g"],
            flipper_length_mm=row["flipper_length_mm"],
        )

    @property
    def band(self):
        """Mass class, using the same thresholds as mass_band."""
        return mass_band(self.body_mass_g)

    def __repr__(self):
        return (
            f"Penguin(species={self.species!r}, island={self.island!r}, "
            f"body_mass_g={self.body_mass_g:.0f}, flipper_length_mm={self.flipper_length_mm:.0f})"
        )

sample = [Penguin.from_row(row) for row in rows[:3]]
for bird in sample:
    print(bird, "->", bird.band)

### Comparing objects

Because `__repr__` is defined, printing a list of penguins is readable. Sorting uses the same
`key=` idea, now reading an attribute.

In [ ]:
heaviest = sorted((Penguin.from_row(r) for r in rows), key=lambda p: p.body_mass_g, reverse=True)[:3]
print("heaviest three:")
for bird in heaviest:
    print(" ", bird)

## Exercises

1. **Parameterise the band.** Change `mass_band` so the two thresholds have defaults but can be
   overridden. Confirm that `mass_band(4000)` and `mass_band(4000, light=3000, heavy=3900)`
   give different answers.
2. **Top five by flipper.** Use `sorted` with a `lambda` to print the five penguins with the
   longest flippers, showing species and flipper length.
3. **Extend the class.** Add a method `is_gentoo(self)` to `Penguin` and a `summary(self)`
   that returns a one-line string. Create one instance and call both.

## Limitations

This is a teaching model, not a data model. `mass_band` returns a label that throws away the
underlying number, and `Penguin` keeps every field as a plain Python object, which is slow and
memory-hungry at scale — pandas or NumPy columns are the right tool there. The class also has no
validation: passing a string where a float is expected fails later with a confusing error. Real
code would validate inputs, and real programs would reuse the packaged `pyutils.mass_band`
rather than redefining it.